# Nikolaisen2022 Plag inventory, grain size, and processing bins

Inventory the individual Plag STL files from `Plag Binary meshes`, filter to particles with metadata `EVSD < 1 um`, estimate volume-equivalent grain size, sort by file size, and assign processing bins.

STL files do not store physical units, so this notebook treats the Nikolaisen2022 coordinates as micrometers by dataset convention and reports characteristic grain sizes in nanometers.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from stl2fem.datasets import add_nikolaisen_stl_metadata, assign_size_bins, nikolaisen_inventory
from stl2fem.quality import estimate_grain_size_stl

DATASET_ROOT = REPO / "data" / "Nikolaisen2022"
OUTPUT_ROOT = REPO / "data" / "Nikolaisen2022_merrill_msh"
REPORT_DIR = OUTPUT_ROOT / "_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PHASES = ("PLAG",)
INPUT_UNIT = "um"
MAX_EVSD_UM = 1.0
REFERENCE_CUBE_EDGES_NM = [10, 50, 100, 200]

## Inventory and Filter

The production default is Plag only. The `<1 um` filter uses the dataset metadata `EVSD (mu)` field, which is the equivalent-sphere diameter reported with the STL geometry.

In [ ]:
inventory = add_nikolaisen_stl_metadata(
    nikolaisen_inventory(DATASET_ROOT, phases=PHASES),
    dataset_root=DATASET_ROOT,
)
filtered = inventory[inventory["metadata_evsd_um"] < MAX_EVSD_UM].copy()
filtered = assign_size_bins(filtered, n_bins=4)
filtered.to_csv(REPORT_DIR / "plag_binary_stl_inventory_evsd_lt_1um.csv", index=False)

print(f"Total Plag meshes: {len(inventory)}")
print(f"Plag meshes with EVSD < {MAX_EVSD_UM} um: {len(filtered)}")
print(filtered.groupby(["phase", "stl_format"]).size())
print(filtered.groupby("size_bin")["stl_size_mib"].agg(["count", "min", "median", "max"]))

filtered[[
    "particle_id", "phase", "stl_format", "metadata_evsd_um",
    "metadata_volume_um3", "stl_size_mib", "size_bin", "source_path",
]]

## Estimate Grain Size

`estimate_grain_size_stl` loads each closed STL surface and computes volume-equivalent characteristic sizes.

- `grain_volume_nm3`: total enclosed STL volume in nm^3.
- `grain_equivalent_cube_edge_nm`: edge length of a cube with the same volume.
- `grain_equivalent_sphere_diameter_nm`: diameter of a sphere with the same volume.
- `grain_bbox_*_nm`: bounding-box dimensions in nm.

The equivalent-cube edge is useful here because the reference markers below are 10, 50, 100, and 200 nm cubes.

In [ ]:
grain_rows = []
for _, row in filtered.iterrows():
    grain_rows.append(estimate_grain_size_stl(row["source_path"], input_unit=INPUT_UNIT))

grain_sizes = pd.DataFrame(grain_rows)
inventory_with_grain = filtered.merge(
    grain_sizes.drop(columns=["input_unit", "input_scale_to_meters"]),
    left_on="source_path",
    right_on="path",
    how="left",
).drop(columns=["path"])

inventory_with_grain.to_csv(
    REPORT_DIR / "plag_binary_stl_inventory_evsd_lt_1um_with_grain_size.csv",
    index=False,
)

columns = [
    "particle_id", "phase", "size_bin", "metadata_evsd_um", "stl_size_mib",
    "grain_volume_nm3", "grain_equivalent_cube_edge_nm",
    "grain_equivalent_sphere_diameter_nm", "grain_bbox_max_nm",
    "source_path",
]
inventory_with_grain[columns]

## Reference Cube Sizes

These are simple cube-volume references. A 100 nm cube has volume `100^3 = 1,000,000 nm^3`. The equivalent-sphere diameter column gives the sphere diameter with the same volume as each reference cube.

In [ ]:
reference_cubes = pd.DataFrame({"reference_cube_edge_nm": REFERENCE_CUBE_EDGES_NM})
reference_cubes["reference_cube_volume_nm3"] = reference_cubes["reference_cube_edge_nm"] ** 3
reference_cubes["equivalent_sphere_diameter_nm"] = (
    6 * reference_cubes["reference_cube_volume_nm3"] / 3.141592653589793
) ** (1 / 3)
reference_cubes

## Size-Bin Summary

The bins are based on STL file size so the plotting/conversion notebooks stay manageable, but the table also shows the physical grain-size range represented in each bin.

In [ ]:
bin_summary = inventory_with_grain.groupby("size_bin").agg(
    count=("particle_id", "count"),
    stl_size_mib_min=("stl_size_mib", "min"),
    stl_size_mib_median=("stl_size_mib", "median"),
    stl_size_mib_max=("stl_size_mib", "max"),
    evsd_um_min=("metadata_evsd_um", "min"),
    evsd_um_median=("metadata_evsd_um", "median"),
    evsd_um_max=("metadata_evsd_um", "max"),
    cube_edge_nm_min=("grain_equivalent_cube_edge_nm", "min"),
    cube_edge_nm_median=("grain_equivalent_cube_edge_nm", "median"),
    cube_edge_nm_max=("grain_equivalent_cube_edge_nm", "max"),
    bbox_max_nm_median=("grain_bbox_max_nm", "median"),
)
bin_summary

## Plots

The first plot shows STL file size, which is what the notebooks are split on. The second plot shows volume-equivalent cube edge length in nm with reference cube sizes overlaid.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

inventory_with_grain["stl_size_mib"].hist(bins=80, ax=axes[0])
axes[0].set_title("Filtered Plag STL file sizes")
axes[0].set_xlabel("STL file size [MiB]")
axes[0].set_ylabel("Count")
axes[0].set_xscale("log")

inventory_with_grain["grain_equivalent_cube_edge_nm"].hist(bins=80, ax=axes[1], color="steelblue")
for edge_nm in REFERENCE_CUBE_EDGES_NM:
    axes[1].axvline(edge_nm, linestyle="--", linewidth=1.5, label=f"{edge_nm} nm cube")
axes[1].set_title("Volume-equivalent grain size, EVSD < 1 um")
axes[1].set_xlabel("Equivalent cube edge length [nm]")
axes[1].set_ylabel("Count")
axes[1].set_xscale("log")
axes[1].legend()

plt.tight_layout()
plt.show()